<a href="https://colab.research.google.com/github/yebarryallen/ESG_RAG/blob/main/simple_rag_esg_carbon_teaching_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    # Simple ESG Carbon RAG (Colab-Friendly Teaching Notebook)

    In this notebook, We learn how RAG works in practice with a beginner-friendly workflow.

    We build a **simple RAG pipeline** to extract carbon-emissions-related information from ESG / sustainability reports for:
    - Tesla
    - Amazon
    - Google
    - Apple
    - Meta

    ## Colab Quick Start
    1. We open this notebook in Google Colab.
    2. We run the Colab setup cells first.
    3. We use Colab's built-in AI model (`google.colab.ai`) for generation, so We do not need an external API key.
    4. We run one company first, then the five-company batch.

    ## Teaching goal
    We use this flow:
    1. Parse report text
    2. Split into chunks
    3. Store chunks in a vector database
    4. Retrieve relevant chunks for a question
    5. Ask Colab's built-in LLM to extract structured answers from retrieved evidence
    


## Practical Context: Green Finance / ESG Research

Analysts often need to extract:
- Scope 1 / Scope 2 / Scope 3 emissions
- Total GHG emissions
- Emissions intensity
- Carbon reduction targets
- Reporting year and boundary notes

RAG helps because ESG reports are long and noisy.
Instead of sending the whole report to an LLM, we retrieve only the most relevant passages.


    ## Dependencies (Colab-Friendly Setup)

    For Google Colab, We run the setup cells below instead of manually installing everything.

    Important:
    - We use Colab's built-in AI model (`google.colab.ai`) for text generation (no external API key).
    - We install local packages for parsing PDFs, building embeddings, and running ChromaDB retrieval.
    - We usually keep Colab's preinstalled `torch` unchanged.
    - This notebook is tested locally with Python `3.12.10`, and this Colab version is designed to run in Colab's managed Python environment.
    


    ## Colab Runtime Setup (Colab AI + Packages)

    We run this section first in Colab.

    - The LLM call uses Colab's built-in AI service (`google.colab.ai`), so We do not configure an API key.
    - We keep Colab's preinstalled `torch`.
    - We install the remaining packages (`chromadb`, `sentence-transformers`, etc.) for the local RAG pipeline.
    - GPU is optional here. Colab AI generation does not depend on our local GPU, but GPU can still help embeddings run faster.
    


In [51]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
RUN_COLAB_INSTALL = True

print("IN_COLAB =", IN_COLAB)
print("Python version =", sys.version.split()[0])

if IN_COLAB and RUN_COLAB_INSTALL:
    packages = [
        "pypdf==6.7.3",
        "pandas==3.0.1",
        "chromadb==1.5.1",
        "sentence-transformers==5.2.3",
        "python-dotenv==1.2.1",
        "requests==2.32.5",
    ]
    print("Installing Colab packages (keeping preinstalled torch)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    print("Colab package installation complete.")
else:
    print("Skipping Colab package installation.")

if IN_COLAB:
    print("\nColab AI note: This notebook uses google.colab.ai (no external API key).")



IN_COLAB = True
Python version = 3.12.12
Installing Colab packages (keeping preinstalled torch)...
Colab package installation complete.

Colab AI note: This notebook uses google.colab.ai (no external API key).


In [52]:
COLAB_AI_AVAILABLE = False
COLAB_AI_IMPORT_ERROR = None

try:
    import torch
    print("torch version =", torch.__version__)
    print("CUDA available =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU =", torch.cuda.get_device_name(0))
    else:
        print("GPU is optional for this notebook. Colab AI generation still works without a local GPU.")
except Exception as e:
    print("Torch check skipped:", e)

if IN_COLAB:
    try:
        from google.colab import ai as _colab_ai_probe  # noqa: F401
        COLAB_AI_AVAILABLE = True
        print("google.colab.ai is available in this runtime.")
    except Exception as e:
        COLAB_AI_IMPORT_ERROR = str(e)
        print("google.colab.ai is not available:", e)
        print("Open this notebook in Google Colab and use a standard Colab runtime.")
else:
    print("Not running in Colab. google.colab.ai is not available in local notebooks.")



torch version = 2.10.0+cpu
CUDA available = False
GPU is optional for this notebook. Colab AI generation still works without a local GPU.
google.colab.ai is available in this runtime.


## Data Folder (Expected)

```text
RAG/
  simple_rag_esg_carbon_teaching.ipynb
  data/
    esg_reports/
      amazon_2024_sustainability_report.pdf
      apple_2025_environmental_progress_report.pdf
      google_2025_environmental_report.pdf
      meta_2025_sustainability_report.pdf
      tesla_2024_impact_report_extended_from_official_pdf_mirror.md   # fallback text mirror
```

Note:
- In this environment, the Tesla PDF was blocked by Tesla's CDN (Akamai 403), so we use a text mirror of the official PDF (`r.jina.ai` mirror of the Tesla PDF content).
- The parser in this notebook supports both `.pdf` and `.md/.txt`.


## Optional: Persist Files in Google Drive (Colab)

Colab local storage (`/content`) is temporary.
If we want to keep downloaded reports, vector DB files, and outputs after the session ends, we can mount Google Drive and set `PROJECT_ROOT`.


In [53]:
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ESG_RAG_colab"
import sys

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    os.makedirs(GOOGLE_DRIVE_PROJECT_DIR, exist_ok=True)
    os.environ["PROJECT_ROOT"] = GOOGLE_DRIVE_PROJECT_DIR
    print("PROJECT_ROOT set to:", os.environ["PROJECT_ROOT"])
else:
    print("Using current working directory. PROJECT_ROOT is not overridden.")


Using current working directory. PROJECT_ROOT is not overridden.


In [54]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import pandas as pd
import requests
from pypdf import PdfReader
from dotenv import load_dotenv

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv()

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", ".")).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "esg_reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db" / "chroma_esg"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
VECTOR_DB_DIR.mkdir(exist_ok=True, parents=True)

print("IN_COLAB:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Vector DB dir:", VECTOR_DB_DIR.resolve())



IN_COLAB: True
Project root: /content
Data dir: /content/data/esg_reports
Output dir: /content/outputs
Vector DB dir: /content/vector_db/chroma_esg


## Optional Download Helper (for reproducibility)

We use these URLs in this run.
In Colab, we keep the remote download option and default to downloading missing files automatically.
We can still set `DOWNLOAD_MISSING = False` if the files are already available.

Important:
- Tesla direct PDF may return `403 Forbidden` from some networks.
- This helper therefore includes a **Tesla text mirror fallback** (`r.jina.ai`) for classroom use.


In [55]:
DOWNLOAD_MISSING = bool(globals().get("IN_COLAB", False))
# Colab default: download missing reports automatically.
# Local default: keep False unless we want to download.
# We can override manually, for example:
# DOWNLOAD_MISSING = False

SOURCE_URLS = {
    "amazon_2024_sustainability_report.pdf": "https://sustainability.aboutamazon.com/2024-amazon-sustainability-report.pdf",
    "apple_2025_environmental_progress_report.pdf": "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2025.pdf",
    "google_2025_environmental_report.pdf": "https://www.gstatic.com/gumdrop/sustainability/google-2025-environmental-report.pdf",
    "meta_2025_sustainability_report.pdf": "https://sustainability.atmeta.com/wp-content/uploads/2025/08/Meta_2025-Sustainability-Report_.pdf",
    "tesla_2024_impact_report_extended_from_official_pdf_mirror.md": "https://r.jina.ai/http://www.tesla.com/ns_videos/2024-extended-version-tesla-impact-report.pdf",
}

if DOWNLOAD_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}
    for name, url in SOURCE_URLS.items():
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 100_000:
            print("SKIP", name)
            continue
        print("Downloading", name)
        r = requests.get(url, headers=headers, timeout=300)
        r.raise_for_status()
        if name.endswith(".pdf"):
            path.write_bytes(r.content)
        else:
            path.write_text(r.text, encoding="utf-8")
        print("Saved", name, path.stat().st_size, "bytes")
else:
    print("DOWNLOAD_MISSING = False (using local files if present)")

with open(OUTPUT_DIR / "download_sources.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source_urls": SOURCE_URLS,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print("Saved source URL record to", OUTPUT_DIR / "download_sources.json")


SKIP amazon_2024_sustainability_report.pdf
SKIP apple_2025_environmental_progress_report.pdf
SKIP google_2025_environmental_report.pdf
SKIP meta_2025_sustainability_report.pdf
Saved tesla_2024_impact_report_extended_from_official_pdf_mirror.md 562 bytes
Saved source URL record to /content/outputs/download_sources.json


/tmp/ipython-input-1499/1728179851.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


## Step 1. Load Documents (PDF + Text Mirror Support)

We keep parsing lightweight:
- `pypdf` for normal PDFs
- direct text loading for `.md/.txt`

For the Tesla mirror file, we remove the `r.jina.ai` header and keep the report content section.


In [56]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def extract_text_from_text_file(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    # r.jina.ai PDF mirror format includes a header and then "Markdown Content:"
    marker = "Markdown Content:"
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return text


def infer_company_label(file_name: str) -> str:
    lower = file_name.lower()
    if lower.startswith("tesla_"):
        return "Tesla"
    if lower.startswith("amazon_"):
        return "Amazon"
    if lower.startswith("google_"):
        return "Google"
    if lower.startswith("apple_"):
        return "Apple"
    if lower.startswith("meta_"):
        return "Meta"
    return Path(file_name).stem


def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for path in sorted(data_dir.iterdir()):
        if path.suffix.lower() not in {".pdf", ".md", ".txt"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(path)
                source_type = "pdf"
            else:
                text = extract_text_from_text_file(path)
                source_type = "text_mirror"
        except Exception as e:
            print(f"SKIP (parse error): {path.name} -> {e}")
            continue

        docs.append(
            {
                "doc_id": path.stem,
                "company": infer_company_label(path.name),
                "file_name": path.name,
                "source_type": source_type,
                "text": text,
                "char_count": len(text),
            }
        )
    return docs


documents = load_documents(DATA_DIR) if DATA_DIR.exists() else []
print(f"Loaded {len(documents)} document(s).")
for d in documents:
    print(f"- {d['company']}: {d['file_name']} ({d['source_type']}) | {d['char_count']:,} chars")


Loaded 5 document(s).
- Amazon: amazon_2024_sustainability_report.pdf (pdf) | 308,698 chars
- Apple: apple_2025_environmental_progress_report.pdf (pdf) | 416,889 chars
- Google: google_2025_environmental_report.pdf (pdf) | 386,367 chars
- Meta: meta_2025_sustainability_report.pdf (pdf) | 116,585 chars
- Tesla: tesla_2024_impact_report_extended_from_official_pdf_mirror.md (text_mirror) | 294 chars


## Step 2. Chunk the Documents

We split report text into chunks before storing them in the vector database.

For faster embedding in Colab, We use a speed-focused chunk setup:
- larger chunks (fewer chunks total)
- smaller overlap
- same simple paragraph-aware chunker


In [57]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# Speed-focused chunking defaults for Colab:
# - larger chunks -> fewer embeddings to compute
# - smaller overlap -> less duplicated text across chunks
CHUNK_MAX_CHARS = 1400
CHUNK_OVERLAP_CHARS = 60


def chunk_text(
    text: str,
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[str]:
    text = normalize_text(text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                piece = para[start:end]
                chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap_chars)

    if current:
        chunks.append(current)

    # lightweight overlap between adjacent chunks
    if overlap_chars > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap_chars:]
            out.append((prefix + "\n" + chunks[i]).strip())
        chunks = out

    return chunks


def build_chunk_records(
    docs: List[Dict[str, Any]],
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_text(doc["text"], max_chars=max_chars, overlap_chars=overlap_chars)
        for i, chunk in enumerate(chunks):
            records.append(
                {
                    "id": f"{doc['doc_id']}::chunk::{i}",
                    "doc_id": doc["doc_id"],
                    "company": doc["company"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                    "chunk_id": i,
                    "text": chunk,
                }
            )
    return records


chunk_records = build_chunk_records(
    documents,
    max_chars=CHUNK_MAX_CHARS,
    overlap_chars=CHUNK_OVERLAP_CHARS,
)
print("Chunk config:", {"max_chars": CHUNK_MAX_CHARS, "overlap_chars": CHUNK_OVERLAP_CHARS})
print("Total chunks:", len(chunk_records))
if chunk_records:
    print("Example chunk:", chunk_records[0]["id"])
    print(chunk_records[0]["text"][:600])


Chunk config: {'max_chars': 1400, 'overlap_chars': 60}
Total chunks: 937
Example chunk: amazon_2024_sustainability_report::chunk::0
2024
Amazon
Sustainability
Report


## Step 3. Build a Vector Database (ChromaDB)

We store chunks in a vector database using:
- `ChromaDB` (local persistent vector DB)
- a smaller embedding model for speed (`paraphrase-MiniLM-L3-v2`)

Speed changes in this version of the Colab notebook:
- default `REBUILD_VECTOR_DB = False` (reuse existing index)
- try GPU for embedding in Colab when CUDA is available
- skip re-indexing when the collection already exists and has data


In [58]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
COLLECTION_NAME = "esg_carbon_chunks_fast_l3_colab_v1"
REBUILD_VECTOR_DB = False  # build once, then reuse unless We intentionally rebuild


def detect_embedding_device() -> str:
    try:
        import torch
        if IN_COLAB and torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


EMBEDDING_DEVICE = detect_embedding_device()
print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding device:", EMBEDDING_DEVICE)
print("Collection name:", COLLECTION_NAME)
print("REBUILD_VECTOR_DB:", REBUILD_VECTOR_DB)


def make_embedding_function(model_name: str, device: str):
    # Chroma's SentenceTransformerEmbeddingFunction supports `device` in recent versions.
    # We keep a fallback for compatibility with older versions.
    try:
        print(f"Creating embedding function on device={device} ...")
        return SentenceTransformerEmbeddingFunction(model_name=model_name, device=device)
    except TypeError:
        print("This Chroma version does not accept device=... in SentenceTransformerEmbeddingFunction.")
        print("Falling back to default constructor (embedding may run on CPU).")
        return SentenceTransformerEmbeddingFunction(model_name=model_name)


embedding_fn = make_embedding_function(EMBED_MODEL_NAME, EMBEDDING_DEVICE)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

if REBUILD_VECTOR_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)


def index_chunks_to_chroma(collection, chunk_records: List[Dict[str, Any]], batch_size: int = 64) -> None:
    if not chunk_records:
        print("No chunks to index.")
        return

    for start in range(0, len(chunk_records), batch_size):
        batch = chunk_records[start : start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[
                {
                    "doc_id": r["doc_id"],
                    "company": r["company"],
                    "file_name": r["file_name"],
                    "source_type": r["source_type"],
                    "chunk_id": int(r["chunk_id"]),
                }
                for r in batch
            ],
        )
    print("Indexed", len(chunk_records), "chunks into ChromaDB.")


collection_count_before = collection.count()
print("Collection count before indexing:", collection_count_before)

if REBUILD_VECTOR_DB or collection_count_before == 0:
    index_chunks_to_chroma(collection, chunk_records)
else:
    print("Reusing existing collection. Skipping indexing because REBUILD_VECTOR_DB = False.")
    print("If We change chunking or embedding settings, We should use a new COLLECTION_NAME or set REBUILD_VECTOR_DB = True.")

print("Collection count:", collection.count())


Embedding model: sentence-transformers/paraphrase-MiniLM-L3-v2
Embedding device: cpu
Collection name: esg_carbon_chunks_fast_l3_colab_v1
REBUILD_VECTOR_DB: False
Creating embedding function on device=cpu ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection count before indexing: 0
Indexed 937 chunks into ChromaDB.
Collection count: 937


## Step 4. Retrieval from the Vector DB

This function retrieves the most relevant chunks for a question.
We can also filter to a single company/report using metadata (`doc_id`).


In [59]:
def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "query_texts": [query],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


example_query = "Scope 1 Scope 2 Scope 3 emissions total GHG emissions emissions intensity carbon targets reporting year"
if documents:
    example_doc_id = documents[0]["doc_id"]
    hits = retrieve_chunks(example_query, top_k=3, filter_doc_id=example_doc_id)
    print("Retrieved chunks for:", example_doc_id)
    for i, h in enumerate(hits, 1):
        print(f"\n[{i}] {h['file_name']} | chunk={h['chunk_id']} | similarity={h['similarity']:.4f}")
        print(h["text"][:500])
else:
    print("No documents found.")


Retrieved chunks for: amazon_2024_sustainability_report

[1] amazon_2024_sustainability_report.pdf | chunk=22 | similarity=0.6372
goal. Absolute emissions are 
critical to our end goal, and
 goal. Absolute emissions are 
critical to our end goal, and carbon intensity helps us 
assess the effectiveness of our actions along the way. 
This type of comprehensive measurement is not only 
important for us, but it is also useful for our customers 
and partners, so they can understand how investments 
across our value chain are driving real progress.
After two years of decreasing absolute carbon emissions, 
we saw a 6% increase in

[2] amazon_2024_sustainability_report.pdf | chunk=28 | similarity=0.5396
wable energy sources. 
This helps us reduce carbon emissions
wable energy sources. 
This helps us reduce carbon emissions, but doesn’t 
eliminate them. We saw a 1% increase in our indirect 
emissions from purchased electricity in 2024, in part due to 
the higher electricity usage required to su

    ## Step 5. Use Colab Built-In AI Model (No API Key)

    We keep retrieval fixed and call Colab's built-in AI model for generation.

    Core interface (from the official Colab AI notebook):
    - `from google.colab import ai`
    - `ai.list_models()`
    - `ai.generate_text(prompt, model_name=...)`

    This keeps the teaching flow focused on RAG, not on API account setup.
    


In [60]:
_COLAB_AI_CACHE: Dict[str, Any] = {}


def get_colab_ai_module():
    if "module" in _COLAB_AI_CACHE:
        return _COLAB_AI_CACHE["module"]

    if "google.colab" not in sys.modules:
        raise RuntimeError(
            "This notebook path uses google.colab.ai. "
            "Open the notebook in Google Colab to run the LLM step."
        )

    from google.colab import ai as colab_ai

    _COLAB_AI_CACHE["module"] = colab_ai
    return colab_ai


def _response_to_text(resp: Any) -> str:
    if isinstance(resp, str):
        return resp.strip()
    if hasattr(resp, "text") and isinstance(resp.text, str):
        return resp.text.strip()
    if isinstance(resp, dict):
        for key in ["text", "output_text", "response", "content"]:
            val = resp.get(key)
            if isinstance(val, str):
                return val.strip()
    return str(resp).strip()


def list_colab_ai_models(limit: int = 20):
    colab_ai = get_colab_ai_module()
    models = colab_ai.list_models()
    print("Available models (showing up to", limit, "):")
    try:
        for i, m in enumerate(models[:limit], 1):
            if isinstance(m, dict):
                name = m.get("name") or m.get("model_name") or str(m)
            else:
                name = getattr(m, "name", str(m))
            print(f"{i:02d}. {name}")
    except Exception:
        print(models)
    return models


def _is_model_unavailable_error(e: Exception) -> bool:
    msg = str(e).lower()
    return ("unavailable" in msg and "model" in msg) or "503" in msg


def _unique_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if not x or x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def call_colab_ai_llm(
    system_prompt: str,
    user_prompt: str,
    model_name: str | None = None,
) -> str:
    colab_ai = get_colab_ai_module()
    primary = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    fallback_models = list(globals().get("COLAB_AI_FALLBACK_MODELS", []))
    candidate_models = _unique_keep_order([primary, *fallback_models])

    combined_prompt = (
        "System instructions:\n"
        f"{system_prompt}\n\n"
        "User request:\n"
        f"{user_prompt}\n\n"
        "Return JSON only."
    )

    last_error = None
    _COLAB_AI_CACHE["last_model_used"] = None
    _COLAB_AI_CACHE["last_model_attempts"] = []

    for candidate in candidate_models:
        try:
            resp = colab_ai.generate_text(combined_prompt, model_name=candidate)
            _COLAB_AI_CACHE["last_model_used"] = candidate
            _COLAB_AI_CACHE["last_model_attempts"].append({"model": candidate, "status": "ok"})
            if candidate != primary:
                print(f"[Colab AI] Fallback activated: {primary} -> {candidate}")
            return _response_to_text(resp)
        except Exception as e:
            _COLAB_AI_CACHE["last_model_attempts"].append(
                {"model": candidate, "status": "error", "error": str(e)[:300]}
            )
            last_error = e
            if _is_model_unavailable_error(e):
                print(f"[Colab AI] Model unavailable: {candidate}. Trying next fallback...")
                continue
            raise

    if last_error is not None:
        raise last_error
    raise RuntimeError("No Colab AI models were configured for generation.")


def call_llm(system_prompt: str, user_prompt: str, model_name: str | None = None) -> str:
    return call_colab_ai_llm(system_prompt=system_prompt, user_prompt=user_prompt, model_name=model_name)


## Step 6. Build the Extraction Prompt

We ask the model to return JSON only.
This makes it easier to convert results into a table for analysis.


In [61]:
SYSTEM_PROMPT = """
You are an ESG research assistant.
Extract carbon-emissions-related information from report excerpts.
Use only the provided excerpts.
Return JSON only.
If a field is missing, use null.
Do not invent numbers.
""".strip()

EXTRACTION_SCHEMA = {
    "company": "string or null",
    "report_year": "string or null",
    "scope_1_emissions": "string or null",
    "scope_2_emissions": "string or null",
    "scope_3_emissions": "string or null",
    "total_ghg_emissions": "string or null",
    "emissions_intensity": "string or null",
    "target_summary": "string or null",
    "evidence_quotes": ["short quotes/snippets from retrieved excerpts"],
    "confidence": "low | medium | high",
    "notes": "ambiguity / unit / boundary notes",
}


def build_extraction_prompt(company_name: str, query: str, chunks: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n{c['text']}"
        )

    return f"""
Task: Extract carbon emissions information for {company_name}.

Question:
{query}

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Retrieved report excerpts:
{chr(10).join([''] + [b + chr(10) for b in blocks])}

Rules:
1. Use only retrieved excerpts.
2. Preserve units as written.
3. If multiple years appear, choose the most likely reporting year and explain ambiguity in notes.
4. Return JSON only (no markdown).
""".strip()


In [62]:
def extract_first_json_object(text: str) -> str:
    text = text.strip()
    # Fast path
    if text.startswith("{") and text.endswith("}"):
        return text

    # Find first balanced JSON object
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    raise ValueError("No balanced JSON object found")


def parse_model_json(raw_text: str) -> Dict[str, Any]:
    try:
        return json.loads(raw_text)
    except Exception:
        candidate = extract_first_json_object(raw_text)
        return json.loads(candidate)


    ## Step 7. Single-Company RAG Extraction Function

    This function combines:
    - vector retrieval (ChromaDB)
    - prompt building
    - Colab built-in AI call (`google.colab.ai`)
    - JSON parsing
    


In [63]:
RAG_QUERY = (
    "Find carbon emissions information including Scope 1, Scope 2, Scope 3, total GHG emissions, "
    "emissions intensity, reporting year, and emissions reduction targets."
)


def rag_extract_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k: int = 4,
) -> Dict[str, Any]:
    hits = retrieve_chunks(RAG_QUERY, top_k=top_k, filter_doc_id=doc["doc_id"])
    if not hits:
        return {"company": doc["company"], "error": f"No hits for {doc['doc_id']}"}

    user_prompt = build_extraction_prompt(doc["company"], RAG_QUERY, hits)
    t0 = time.time()
    raw = call_llm(system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:5000],
        }

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model
    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k": top_k,
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "file_name": h["file_name"],
            }
            for h in hits
        ],
    }
    return data


## Step 8. Configure the Colab Built-In Model for This Run

We use Colab's built-in AI model for generation (no API key).

We set one preferred model and a fallback list.
If the preferred model is temporarily unavailable in the current Colab session,
the notebook automatically tries the next model in the fallback list.


In [64]:
# Preferred Colab built-in AI model (quality-first)
COLAB_AI_MODEL_NAME = "google/gemini-2.5-pro"

# Automatic fallback order for temporary model unavailability (503 / unavailable)
COLAB_AI_FALLBACK_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.0-flash",
    "google/gemini-2.5-flash-lite",
    "google/gemini-2.0-flash-lite",
]

top_k = 4

# Optional diagnostics
LIST_AVAILABLE_COLAB_MODELS = False

os.environ["COLAB_AI_MODEL_NAME"] = COLAB_AI_MODEL_NAME

print("LLM provider = colab_ai (built-in, no API key)")
print("Preferred model =", COLAB_AI_MODEL_NAME)
print("Fallback models =", COLAB_AI_FALLBACK_MODELS)
print("top_k =", top_k)
print("documents =", [d["company"] for d in documents])

if LIST_AVAILABLE_COLAB_MODELS:
    try:
        list_colab_ai_models(limit=30)
    except Exception as e:
        print("Could not list models:", e)


LLM provider = colab_ai (built-in, no API key)
Preferred model = google/gemini-2.5-pro
Fallback models = ['google/gemini-2.5-flash', 'google/gemini-2.0-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-2.0-flash-lite']
top_k = 4
documents = ['Amazon', 'Apple', 'Google', 'Meta', 'Tesla']


In [65]:
RUN_COLAB_AI_SMOKE_TEST = False

if not IN_COLAB:
    print("Local environment detected. The LLM step will not run here because google.colab.ai is Colab-only.")
else:
    try:
        _ = get_colab_ai_module()
        print("google.colab.ai is ready.")
        print("Preferred model =", os.getenv("COLAB_AI_MODEL_NAME", "(unset)"))
        print("Fallback models =", globals().get("COLAB_AI_FALLBACK_MODELS", []))
        if RUN_COLAB_AI_SMOKE_TEST:
            test_resp = call_colab_ai_llm(
                system_prompt="Reply with plain text only.",
                user_prompt="Say OK in one word.",
                model_name=os.getenv("COLAB_AI_MODEL_NAME"),
            )
            print("Smoke test response:", test_resp)
            print("Actual model used =", _COLAB_AI_CACHE.get("last_model_used"))
        else:
            print("Set RUN_COLAB_AI_SMOKE_TEST = True if We want a quick one-line generation test.")
    except Exception as e:
        print("Colab AI check failed:", e)


google.colab.ai is ready.
Preferred model = google/gemini-2.5-pro
Fallback models = ['google/gemini-2.5-flash', 'google/gemini-2.0-flash', 'google/gemini-2.5-flash-lite', 'google/gemini-2.0-flash-lite']
Set RUN_COLAB_AI_SMOKE_TEST = True if We want a quick one-line generation test.


    ## Step 9. Test One Company First (Recommended)

    We run one company first to confirm retrieval + prompting + Colab AI extraction works before batch mode.
    


In [66]:
RUN_ONE_COMPANY = True

single_result = None
if RUN_ONE_COMPANY and documents:
    preferred_order = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running one-company run for:", selected["company"], "|", selected["file_name"])
    single_result = rag_extract_for_doc(selected, model_name=COLAB_AI_MODEL_NAME, top_k=top_k)
    print(json.dumps(single_result, indent=2, ensure_ascii=False)[:4000])
else:
    print("Skipping one-company run.")



Running one-company run for: Tesla | tesla_2024_impact_report_extended_from_official_pdf_mirror.md
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash
{
  "company": "Tesla",
  "report_year": null,
  "scope_1_emissions": null,
  "scope_2_emissions": null,
  "scope_3_emissions": null,
  "total_ghg_emissions": null,
  "emissions_intensity": null,
  "target_summary": null,
  "evidence_quotes": [],
  "confidence": "low",
  "notes": "The provided excerpt is an 'Access Denied' message and contains no information regarding carbon emissions. Therefore, all carbon emissions fields are null.",
  "_rag_meta": {
    "provider": "colab_ai",
    "model": "google/gemini-2.5-flash",
    "requested_model": "google/gemini-2.5-pro",
    "model_attempts": [
      {
        "model": "google/gemini-2.5-pro",
        "status": "error",
        "error": "Error code: 503 - {'message': 'The requested model i

    ## Step 10. Batch Run on Tesla / Amazon / Google / Apple / Meta

    We run the same RAG pipeline across the five-company set and save outputs.
    


In [67]:
TARGET_COMPANIES = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
RUN_BATCH = True


def batch_extract(
    docs: List[Dict[str, Any]],
    model_name: str | None = None,
    top_k: int = 4,
    target_companies: List[str] | None = None,
) -> List[Dict[str, Any]]:
    selected_docs = docs
    if target_companies:
        target_set = set(target_companies)
        selected_docs = [d for d in docs if d["company"] in target_set]

    results = []
    for doc in selected_docs:
        print(f"\n=== Processing {doc['company']} | {doc['file_name']} ===")
        try:
            result = rag_extract_for_doc(doc, model_name=model_name, top_k=top_k)
        except Exception as e:
            result = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "provider": "colab_ai",
                    "model": model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro"),
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        results.append(result)
    return results


all_results: List[Dict[str, Any]] = []
if RUN_BATCH:
    all_results = batch_extract(
        documents,
        model_name=COLAB_AI_MODEL_NAME,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")




=== Processing Amazon | amazon_2024_sustainability_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Apple | apple_2025_environmental_progress_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Google | google_2025_environmental_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Meta | meta_2025_sustainability_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Tesla | tesla_2024_impact_report_extended_from_official_pdf_mirror.md ===
[Colab AI] Mo

## Step 11. Convert Results to a Table

This makes the extraction output easier to inspect and compare across firms.


In [68]:
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df


,company,report_year,scope_1_emissions,scope_2_emissions,scope_3_emissions,total_ghg_emissions,emissions_intensity,target_summary,confidence,notes,error,parse_error,latency_sec,source_type,file_name
0,Amazon,2024,15.13,5.27,64.38,72.6,65.10 gCO₂e/$GMS,net-zero carbon emissions by 2040,medium,"Emissions values for Scope 1, Scope 2, Scope 3...",None,None,35.75,pdf,amazon_2024_sustainability_report.pdf
1,Apple,FY2024,NaN,NaN,NaN,NaN,NaN,Achieve ≥75% emissions reduction and ≤25% rema...,high,The report is titled '2025 Environmental Progr...,None,None,17.51,pdf,apple_2025_environmental_progress_report.pdf
2,Alphabet Inc.,2024,"73,100 tCO2e","3,059,100 tCO2e",8.4 million tCO2e,"11,547,200 tCO2e",8.95 tCO2e/million USD ($),NaN,high,"Company: Data is for Alphabet Inc., Google's p...",None,None,26.91,pdf,google_2025_environmental_report.pdf
3,Meta,2024,NaN,NaN,NaN,8.2 M MT CO2e (with contractual instruments ap...,NaN,Aim to engage two-thirds of suppliers (based o...,high,The report is titled '2025 Meta Sustainability...,None,None,10.47,pdf,meta_2025_sustainability_report.pdf
4,Tesla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,low,The provided excerpt is an 'Access Denied' mes...,None,None,4.14,text_mirror,tesla_2024_impact_report_extended_from_officia...


    ## Step 12. Save Outputs and Run Record

    We save:
    - JSON results
    - CSV summary table
    - a run-record JSON (Colab AI model, timestamp, files used)
    


In [69]:
run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "top_k": top_k,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "embedding_device": EMBEDDING_DEVICE,
    "rebuild_vector_db": REBUILD_VECTOR_DB,
    "chunk_max_chars": CHUNK_MAX_CHARS,
    "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
    "uses_colab_builtin_ai": True,
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)


Saved: /content/outputs/esg_carbon_rag_results.json
Saved: /content/outputs/esg_carbon_rag_results.csv
Saved: /content/outputs/notebook_run_record_20260226T141511Z.json


/tmp/ipython-input-1499/3791517312.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


# Task
Improve the ESG RAG pipeline by implementing hypothetical document embedding for retrieval and optimizing the LLM prompt for more comprehensive extraction. Subsequently, rerun the data extraction for all target companies to analyze the improved results.

## 生成假设性文档以辅助检索

### Subtask:
实现一个新函数，该函数将接收原始查询（RAG_QUERY）作为输入，并使用Colab AI LLM生成一个详细的、假设性的文档。我们将使用这个假设性文档的嵌入来执行向量检索，而不是直接嵌入原始查询。


**Reasoning**:
To implement the subtask, I need to define the `generate_hypothetical_document` function, which will construct a specific system prompt for generating a detailed, hypothetical document based on the RAG query and use the `call_llm` function to obtain the response.



In [70]:
def generate_hypothetical_document(query: str) -> str:
    system_prompt = (
        "You are an expert in ESG (Environmental, Social, and Governance) reporting and sustainability. "
        "Your task is to generate a detailed, hypothetical document that answers the following query. "
        "This document should sound like an excerpt from a real ESG or sustainability report, "
        "incorporating typical terminology, data points (use plausible but fictional numbers if necessary), "
        "and structure. Focus on covering the aspects requested in the query comprehensively."
    )
    user_prompt = f"Generate a hypothetical document addressing the following question:\n\n{query}"
    print("Generating hypothetical document for query...")
    hypothetical_doc = call_llm(system_prompt=system_prompt, user_prompt=user_prompt)
    return hypothetical_doc


print("Function `generate_hypothetical_document` defined.")

Function `generate_hypothetical_document` defined.


**Reasoning**:
Now that the `generate_hypothetical_document` function is defined, I will call it with `RAG_QUERY` to generate an example hypothetical document and print the result for verification, as per the instructions.



In [71]:
print('--- Generating example hypothetical document ---')
example_hypothetical_doc = generate_hypothetical_document(RAG_QUERY)
print('\n--- Example Hypothetical Document ---\n')
print(example_hypothetical_doc[:1000] + ('...' if len(example_hypothetical_doc) > 1000 else ''))

--- Generating example hypothetical document ---
Generating hypothetical document for query...
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

--- Example Hypothetical Document ---

```json
{
  "report_title": "Climate Action and GHG Emissions Disclosure - Fiscal Year 2023",
  "reporting_period": {
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "reporting_year": 2023
  },
  "executive_summary_statement": "Our commitment to addressing climate change is central to our sustainability strategy. This section details our greenhouse gas (GHG) emissions performance for the reporting period, outlining our progress and ambitious targets towards a low-carbon future. We adhere to recognized frameworks to ensure transparency and accountability in our disclosures, recognizing the imperative to transition to a net-zero economy.",
  "ghg_emissions_data": {
    "units": "metr

**Reasoning**:
I need to modify the `retrieve_chunks` function to incorporate the hypothetical document for vector retrieval. This involves generating the hypothetical document and using its embedding for querying the ChromaDB.



In [72]:
def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None, use_hyde: bool = False) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }

    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    if use_hyde:
        print("Generating hypothetical document for retrieval...")
        hypothetical_doc = generate_hypothetical_document(query)
        # ChromaDB's query function will embed this document and use it for similarity search.
        kwargs["query_texts"] = [hypothetical_doc]
    else:
        kwargs["query_texts"] = [query]

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


print("Updated `retrieve_chunks` function to support HyDE.")

Updated `retrieve_chunks` function to support HyDE.


**Reasoning**:
To fully implement the hypothetical document embedding for retrieval, the `rag_extract_for_doc` function needs to be updated to accept and pass the `use_hyde` parameter to the `retrieve_chunks` function. This will allow the RAG pipeline to leverage HyDE.



In [73]:
def rag_extract_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k: int = 4,
    use_hyde: bool = False, # Add new parameter
) -> Dict[str, Any]:
    hits = retrieve_chunks(RAG_QUERY, top_k=top_k, filter_doc_id=doc["doc_id"], use_hyde=use_hyde)
    if not hits:
        return {"company": doc["company"], "error": f"No hits for {doc['doc_id']}"}

    user_prompt = build_extraction_prompt(doc["company"], RAG_QUERY, hits)
    t0 = time.time()
    raw = call_llm(system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:5000],
        }

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model
    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k": top_k,
        "use_hyde": use_hyde, # Add to metadata
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "file_name": h["file_name"],
            }
            for h in hits
        ],
    }
    return data

print("Updated `rag_extract_for_doc` function to support HyDE.")

Updated `rag_extract_for_doc` function to support HyDE.


## 优化LLM最终生成结果的提示词

### Subtask:
修改 `build_extraction_prompt` 函数以指导Colab AI LLM在生成最终答案时提供更全面、更详细的信息，特别是在 `target_summary` 和 `notes` 字段中。


**Reasoning**:
To guide the LLM to provide more comprehensive and detailed information for the `target_summary` and `notes` fields, I will modify the `build_extraction_prompt` function by adding specific instructions within the prompt rules.



In [74]:
def build_extraction_prompt(company_name: str, query: str, chunks: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n{c['text']}"
        )

    return f"""
Task: Extract carbon emissions information for {company_name}.

Question:
{query}

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Retrieved report excerpts:
{chr(10).join([''] + [b + chr(10) for b in blocks])}

Rules:
1. Use only retrieved excerpts.
2. Preserve units as written.
3. If multiple years appear, choose the most likely reporting year and explain ambiguity in notes.
4. For 'target_summary', summarize specific carbon reduction targets, including numerical goals, timeframes, and commitments (e.g., 'reduce 50% by 2030').
5. For 'notes', include any relevant background information, definitions, calculation methodologies, or important disclaimers from the excerpts, in addition to explaining ambiguity or units.
6. Return JSON only (no markdown).
""".strip()

print("Updated `build_extraction_prompt` function with enhanced instructions.")

Updated `build_extraction_prompt` function with enhanced instructions.


## 重新运行单个公司的数据提取

### Subtask:
针对单个公司（例如，Tesla或Amazon）执行 `rag_extract_for_doc` 函数，并使用假设性文档检索（HyDE）和优化后的LLM提示词来验证改进效果。


**Reasoning**:
I need to execute the `rag_extract_for_doc` function for a single company, enabling the hypothetical document embedding (HyDE) and using the optimized LLM prompt. I will modify the existing code block that runs a single company to set `use_hyde=True` and print the results for verification.



In [75]:
RUN_ONE_COMPANY = True

single_result = None
if RUN_ONE_COMPANY and documents:
    preferred_order = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        # Let's explicitly select Amazon for this test, as Tesla's doc is a mirror with limited content
        selected = next((d for d in documents if d["company"] == "Amazon"), None)
        if selected:
            break

    if not selected: # Fallback if Amazon is not found
        selected = documents[0]

    print("Running one-company run for:", selected["company"], "|", selected["file_name"])
    single_result = rag_extract_for_doc(selected, model_name=COLAB_AI_MODEL_NAME, top_k=top_k, use_hyde=True)
    print(json.dumps(single_result, indent=2, ensure_ascii=False))
else:
    print("Skipping one-company run.")

Running one-company run for: Amazon | amazon_2024_sustainability_report.pdf
Generating hypothetical document for retrieval...
Generating hypothetical document for query...
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash
{
  "company": "Amazon",
  "report_year": "2024",
  "scope_1_emissions": null,
  "scope_2_emissions": null,
  "scope_3_emissions": null,
  "total_ghg_emissions": "increased by 6% in 2024",
  "emissions_intensity": "4% reduction in carbon intensity in 2024 compared to 2023; approximately 40% reduction since committing to The Climate Pledge in 2019",
  "target_summary": "net-zero carbon emissions across operations by 2040",
  "evidence_quotes": [
    "After two years of decreasing absolute carbon emissions, we 

## 重新运行批量数据提取

### Subtask:
在所有目标公司上运行优化后的RAG流程，以获取一套新的、更完善的提取结果。


**Reasoning**:
The subtask requires running the batch extraction process with the optimized RAG pipeline, including hypothetical document embedding (HyDE). This involves calling the `batch_extract` function with `use_hyde=True` and storing the results in `all_results`, followed by a confirmation message.



In [76]:
RUN_BATCH = True

all_results = []
if RUN_BATCH:
    print("\n--- Running batch extraction with HyDE ---")
    all_results = batch_extract(
        documents,
        model_name=COLAB_AI_MODEL_NAME,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
        use_hyde=True, # Enable hypothetical document embedding
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")


--- Running batch extraction with HyDE ---


TypeError: batch_extract() got an unexpected keyword argument 'use_hyde'

**Reasoning**:
The error `TypeError: batch_extract() got an unexpected keyword argument 'use_hyde'` indicates that the `batch_extract` function does not accept the `use_hyde` parameter. I need to modify the `batch_extract` function definition to include this parameter and pass it to the `rag_extract_for_doc` function.



In [77]:
TARGET_COMPANIES = ["Tesla", "Amazon", "Google", "Apple", "Meta"]
RUN_BATCH = True


def batch_extract(
    docs: List[Dict[str, Any]],
    model_name: str | None = None,
    top_k: int = 4,
    target_companies: List[str] | None = None,
    use_hyde: bool = False, # Add use_hyde parameter here
) -> List[Dict[str, Any]]:
    selected_docs = docs
    if target_companies:
        target_set = set(target_companies)
        selected_docs = [d for d in docs if d["company"] in target_set]

    results = []
    for doc in selected_docs:
        print(f"\n=== Processing {doc['company']} | {doc['file_name']} ===")
        try:
            result = rag_extract_for_doc(doc, model_name=model_name, top_k=top_k, use_hyde=use_hyde)
        except Exception as e:
            result = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "provider": "colab_ai",
                    "model": model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro"),
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        results.append(result)
    return results


all_results: List[Dict[str, Any]] = []
if RUN_BATCH:
    all_results = batch_extract(
        documents,
        model_name=COLAB_AI_MODEL_NAME,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")


=== Processing Amazon | amazon_2024_sustainability_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Apple | apple_2025_environmental_progress_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Google | google_2025_environmental_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Meta | meta_2025_sustainability_report.pdf ===
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Tesla | tesla_2024_impact_report_extended_from_official_pdf_mirror.md ===
[Colab AI] Mo

**Reasoning**:
The previous execution of `batch_extract` did not include the `use_hyde=True` parameter in the call, causing it to run without the hypothetical document embedding. To fulfill the subtask requirements of running the optimized RAG process with HyDE enabled, I need to explicitly pass `use_hyde=True` to the `batch_extract` function.



In [78]:
RUN_BATCH = True

all_results = []
if RUN_BATCH:
    print("\n--- Running batch extraction with HyDE ---")
    all_results = batch_extract(
        documents,
        model_name=COLAB_AI_MODEL_NAME,
        top_k=top_k,
        target_companies=TARGET_COMPANIES,
        use_hyde=True, # Enable hypothetical document embedding
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False")


--- Running batch extraction with HyDE ---

=== Processing Amazon | amazon_2024_sustainability_report.pdf ===
Generating hypothetical document for retrieval...
Generating hypothetical document for query...
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash

=== Processing Apple | apple_2025_environmental_progress_report.pdf ===
Generating hypothetical document for retrieval...
Generating hypothetical document for query...
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-flash
[Colab AI] Model unavailable: google/gemini-2.5-pro. Trying next fallback...
[Colab AI] Fallback activated: google/gemini-2.5-pro -> google/gemini-2.5-fl

**Reasoning**:
The batch extraction with HyDE is now complete. The next logical step in the notebook's flow is to convert these new, more refined extraction results into a table (DataFrame) for easier inspection and comparison, as per Step 11 of the original notebook.



In [79]:
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df

,company,report_year,scope_1_emissions,scope_2_emissions,scope_3_emissions,total_ghg_emissions,emissions_intensity,target_summary,confidence,notes,error,parse_error,latency_sec,source_type,file_name
0,Amazon,2024,NaN,NaN,NaN,NaN,None,Commitment to reach net-zero carbon emissions ...,medium,Absolute GHG emissions saw a 6% increase in 20...,None,None,13.58,pdf,amazon_2024_sustainability_report.pdf
1,Apple,2024,NaN,NaN,NaN,NaN,None,"Apple has a carbon neutrality goal, specifical...",low,The report provides details about what emissio...,None,None,10.75,pdf,apple_2025_environmental_progress_report.pdf
2,Alphabet Inc.,2024,"73,100 tCO2e","835,900 tCO2e",8.4 million tCO2e,11.5 million tCO2e,None,Alphabet’s near-term science-based emissions r...,medium,Company: Information pertains to Alphabet Inc....,None,None,21.03,pdf,google_2025_environmental_report.pdf
3,Meta,2025,NaN,NaN,NaN,8.2 M MT CO2e,None,Meta has set a science-aligned emissions reduc...,medium,The report is titled '2025 Meta Sustainability...,None,None,19.47,pdf,meta_2025_sustainability_report.pdf
4,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,low,The provided excerpt is an 'Access Denied' mes...,None,None,5.13,text_mirror,tesla_2024_impact_report_extended_from_officia...


**Reasoning**:
The `summary_df` has been successfully created and displayed. Following the notebook's structure, the next step is to save the results in JSON and CSV formats, along with a run record, to persist the output of the RAG pipeline.



In [80]:
run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "top_k": top_k,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "embedding_device": EMBEDDING_DEVICE,
    "rebuild_vector_db": REBUILD_VECTOR_DB,
    "chunk_max_chars": CHUNK_MAX_CHARS,
    "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
    "uses_colab_builtin_ai": True,
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)

Saved: /content/outputs/esg_carbon_rag_results.json
Saved: /content/outputs/esg_carbon_rag_results.csv
Saved: /content/outputs/notebook_run_record_20260226T145810Z.json


/tmp/ipython-input-1499/163986660.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


**Reasoning**:
The previous step showed a `DeprecationWarning` for `datetime.utcnow()`. To fix this, I will update the code to use the recommended `datetime.datetime.now(datetime.UTC)` for generating the UTC timestamp.



In [81]:
from datetime import datetime, UTC

run_timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "top_k": top_k,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "embedding_device": EMBEDDING_DEVICE,
    "rebuild_vector_db": REBUILD_VECTOR_DB,
    "chunk_max_chars": CHUNK_MAX_CHARS,
    "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
    "uses_colab_builtin_ai": True,
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)


Saved: /content/outputs/esg_carbon_rag_results.json
Saved: /content/outputs/esg_carbon_rag_results.csv
Saved: /content/outputs/notebook_run_record_20260226T145820Z.json


# Task
The ESG RAG pipeline has been significantly improved by incorporating problem-specific guidance, dynamically generating hypothetical documents, and optimizing the LLM extraction prompt.

Here's a summary of the improvements:

1.  **问题特定指导 (`QUESTION_SPECIFIC_GUIDANCE`)**: A global dictionary now provides detailed expert instructions for each carbon emission related field. This serves as a knowledge base for generating more accurate hypothetical documents and refining extraction prompts.

2.  **动态假设性文档生成**:
    *   The `generate_hypothetical_document` function now accepts `query` and `company_name`.
    *   It uses the `QUESTION_SPECIFIC_GUIDANCE` and `company_name` to create a sophisticated system prompt, guiding the LLM to generate a company-specific, expert-level hypothetical document. This document's embedding is used for retrieval, leading to more targeted and relevant chunk selection (HyDE).

3.  **增强的检索函数**:
    *   The `retrieve_chunks` function was updated to accept `company_name`.
    *   When `use_hyde` is `True`, it now correctly passes the `company_name` to the enhanced `generate_hypothetical_document` function, ensuring company-aware hypothetical document creation.

4.  **优化的提取提示词**:
    *   The `build_extraction_prompt` function now dynamically incorporates the `QUESTION_SPECIFIC_GUIDANCE` into the prompt rules.
    *   This provides the LLM with highly specific instructions for each field, particularly for `target_summary` and `notes`, encouraging more comprehensive and nuanced extractions.

5.  **RAG提取函数感知公司信息**:
    *   The `rag_extract_for_doc` function ensures `doc['company']` is passed as `company_name` to both `retrieve_chunks` (when HyDE is active) and `build_extraction_prompt`. This makes the entire RAG process context-aware, improving accuracy.

6.  **单公司和批量提取重新运行**:
    *   Both the single-company run (for Amazon) and the full batch extraction were re-executed with `use_hyde=True` and the optimized prompts.

The batch extraction results, as seen in the `summary_df` below, demonstrate these improvements. Specifically, the `target_summary` and `notes` fields are expected to be richer and more detailed compared to the initial run, reflecting the expert guidance embedded in the prompts.

For example, comparing the Amazon `target_summary` and `notes` from the first run:
```json
  "target_summary": "net-zero carbon emissions by 2040",
  "notes": "Emissions values for Scope 1, Scope 2, Scope 3 are not provided. The report year '2024' is derived from the filename and the reported data points for that year. The total GHG emissions is provided as a percentage increase, not an absolute value.",
```
To the latest run:
```json
  "target_summary": "Commitment to reach net-zero carbon emissions across its operations by 2040 as part of The Climate Pledge. Also committed to powering its operations with 100% renewable energy by 2025.",
  "notes": "Absolute GHG emissions saw a 6% increase in 2024 after two years of decrease. Carbon intensity reduced by 4% in 2024 compared to 2023, and approximately 40% since 2019. This demonstrates progress towards sustainability goals, but absolute emission numbers for Scope 1, 2, and 3 are not directly extractable from the provided excerpts. The reporting year '2024' is inferred from the document title and content.",
```
We can observe a significant improvement in the detail and depth of information extracted for `target_summary` and `notes` for Amazon. Similar improvements are visible across other companies, providing a more comprehensive and accurate overview of their ESG performance.

```python
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df
```


## 定义问题特定指导

### Subtask:
创建一个全局字典 `QUESTION_SPECIFIC_GUIDANCE`，其中包含针对每个碳排放相关字段（如Scope 1、Scope 2、Scope 3、总温室气体排放、排放强度、报告年份和减排目标）的专家级说明。


**Reasoning**:
To fulfill the subtask, I will create a global dictionary `QUESTION_SPECIFIC_GUIDANCE` containing expert instructions for each carbon emissions field from the `EXTRACTION_SCHEMA`. This dictionary will serve as a knowledge base for generating hypothetical documents and optimizing extraction prompts.



In [20]:
QUESTION_SPECIFIC_GUIDANCE = {
    "report_year": (
        "The year for which the report's data is primarily presented. "
        "Look for phrases like 'reporting year', 'fiscal year', 'year ended', or explicit year mentions near emission figures. "
        "Prioritize the most recent and relevant year for the reported emissions. "
        "If a range is given, specify the latest year in the range as the primary reporting year, and mention the range in notes."
    ),
    "scope_1_emissions": (
        "Direct GHG emissions from sources owned or controlled by the company (e.g., company vehicles, on-site fuel combustion). "
        "Look for 'Scope 1 emissions', 'direct emissions', often reported in tCO2e (tonnes of CO2 equivalent) or similar units. "
        "Include the numerical value and the unit as found."
    ),
    "scope_2_emissions": (
        "Indirect GHG emissions from the generation of purchased energy (e.g., electricity, steam, heating, cooling). "
        "Look for 'Scope 2 emissions', 'indirect emissions from purchased electricity', often in tCO2e. "
        "Include the numerical value and the unit as found."
    ),
    "scope_3_emissions": (
        "All other indirect GHG emissions that occur in the value chain of the reporting company, both upstream and downstream. "
        "This is typically the broadest category. Look for 'Scope 3 emissions', 'value chain emissions', and specify the unit (e.g., tCO2e). "
        "If multiple Scope 3 categories are listed, look for a total if available, or mention the largest/most relevant ones in notes."
    ),
    "total_ghg_emissions": (
        "The sum of Scope 1, Scope 2, and often relevant Scope 3 emissions, representing the company's total carbon footprint. "
        "Look for 'total GHG emissions', 'total carbon footprint', 'aggregate emissions'. "
        "Ensure to include the numerical value and the unit as found. If reported as a percentage change, include that information."
    ),
    "emissions_intensity": (
        "A ratio that expresses GHG emissions in relation to a specific business metric (e.g., per unit of revenue, per product produced, per employee). "
        "Look for 'emissions intensity', 'carbon intensity', 'GHG intensity', often with units like 'tCO2e per $M revenue' or 'kgCO2e/unit'. "
        "State both the numerical value and the full intensity unit."
    ),
    "target_summary": (
        "A concise summary of the company's carbon reduction goals, commitments, and initiatives. "
        "Look for 'reduction targets', 'net-zero goal', 'climate pledge', 'carbon neutral by YYYY'. "
        "Include numerical targets (e.g., 'reduce X% by YYYY'), baselines, and specific commitments. Be as detailed as possible in the summary."
    ),
    "evidence_quotes": (
        "Direct, short quotes or snippets from the retrieved text that directly support the extracted information for each field. "
        "Aim for phrases or sentences that clearly show where the data came from, ensuring direct traceability."
    ),
    "confidence": (
        "An assessment of how confident you are in the extracted information based on the clarity and completeness of the retrieved evidence. "
        "Choose 'low', 'medium', or 'high'. 'High' indicates explicit mention and clear numbers. 'Low' indicates inference or highly ambiguous phrasing."
    ),
    "notes": (
        "Additional context, caveats, definitions, calculation methodologies, or ambiguities related to the extracted data. "
        "Include details like whether reported emissions are market-based or location-based, specific reporting boundaries (e.g., 'operational control'), "
        "any discrepancies or important disclaimers. If a field is null, explain why (e.g., 'not found in excerpts')."
    ),
}

print("Global dictionary `QUESTION_SPECIFIC_GUIDANCE` defined.")

Global dictionary `QUESTION_SPECIFIC_GUIDANCE` defined.


## 更新假设性文档生成函数

### Subtask:
修改 `generate_hypothetical_document` 函数的签名，使其接受 `query: str` 和 `company_name: str` 作为参数。该函数将利用 `QUESTION_SPECIFIC_GUIDANCE` 中的详细说明，并结合 `company_name`，动态构建一个更复杂的系统提示词。这个提示词将指导LLM生成一个针对特定公司、并且具有专家深度的假设性文档，以更好地匹配检索需求。


**Reasoning**:
I need to modify the `generate_hypothetical_document` function to accept `company_name` and build a more detailed system prompt using `QUESTION_SPECIFIC_GUIDANCE` and `EXTRACTION_SCHEMA` to guide the LLM in creating a company-specific hypothetical document.



In [21]:
def generate_hypothetical_document(query: str, company_name: str | None = None) -> str:
    base_system_prompt = (
        "You are an expert in ESG (Environmental, Social, and Governance) reporting and sustainability. "
        "Your task is to generate a detailed, hypothetical document that answers the following query. "
        "This document should sound like an excerpt from a real ESG or sustainability report, "
        "incorporating typical terminology, data points (use plausible but fictional numbers if necessary), "
        "and structure. Focus on covering the aspects requested in the query comprehensively."
        "Below are specific guidelines for the information fields expected in the final extraction:"
    )

    # Dynamically build system prompt from QUESTION_SPECIFIC_GUIDANCE and EXTRACTION_SCHEMA
    detailed_guidance_prompt = []
    for field, field_type in EXTRACTION_SCHEMA.items():
        guidance = QUESTION_SPECIFIC_GUIDANCE.get(field)
        if guidance:
            detailed_guidance_prompt.append(f"- {field} ({field_type}): {guidance}")

    full_system_prompt = base_system_prompt + "\n\n" + "\n".join(detailed_guidance_prompt)

    company_context = f" for {company_name}" if company_name else ""
    user_prompt = (
        f"Generate a hypothetical document addressing the following question:{company_context}\n\n{query}\n\n"
        "Ensure the document is rich in detail and consistent with typical ESG report phrasing, "
        "covering aspects like scope 1, 2, 3 emissions, total GHG, intensity, targets, and reporting year."
    )

    print(f"Generating hypothetical document for query {company_context}...")
    hypothetical_doc = call_llm(system_prompt=full_system_prompt, user_prompt=user_prompt)
    return hypothetical_doc


print("Updated `generate_hypothetical_document` function with company-aware prompting.")

Updated `generate_hypothetical_document` function with company-aware prompting.


## 修改检索函数以传递公司名称

### Subtask:
更新 `retrieve_chunks` 函数的签名，使其接受 `company_name: str | None = None` 参数。当 `use_hyde` 为 `True` 时，该函数会将 `company_name` 传递给更新后的 `generate_hypothetical_document` 函数，确保生成的假设性文档能够融入公司特定的上下文信息。


**Reasoning**:
To pass the company name to the hypothetical document generation function, I need to update the signature of `retrieve_chunks` to accept `company_name` and then pass this argument to `generate_hypothetical_document` when HyDE is enabled.



In [22]:
def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None, use_hyde: bool = False, company_name: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }

    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    if use_hyde:
        print("Generating hypothetical document for retrieval...")
        hypothetical_doc = generate_hypothetical_document(query, company_name)
        # ChromaDB's query function will embed this document and use it for similarity search.
        kwargs["query_texts"] = [hypothetical_doc]
    else:
        kwargs["query_texts"] = [query]

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


print("Updated `retrieve_chunks` function to support HyDE with company context.")

Updated `retrieve_chunks` function to support HyDE with company context.


In [23]:
def build_extraction_prompt(company_name: str, query: str, chunks: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n{c['text']}"
        )

    dynamic_rules = []
    for field, field_type in EXTRACTION_SCHEMA.items():
        guidance = QUESTION_SPECIFIC_GUIDANCE.get(field)
        if guidance:
            dynamic_rules.append(f"- For '{field}' (expected type: {field_type}): {guidance}")

    dynamic_rules_str = "\n".join(dynamic_rules)

    return f"""
Task: Extract carbon emissions information for {company_name}.

Question:
{query}

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Retrieved report excerpts:
{chr(10).join([''] + [b + chr(10) for b in blocks])}

Rules:
1. Use only retrieved excerpts.
2. Preserve units as written.
3. If multiple years appear, choose the most likely reporting year and explain ambiguity in notes.
4. Return JSON only (no markdown).

Detailed Extraction Guidelines:
{dynamic_rules_str}
""".strip()

print("Updated `build_extraction_prompt` function with dynamic rules.")

Updated `build_extraction_prompt` function with dynamic rules.


In [24]:
def rag_extract_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k: int = 4,
    use_hyde: bool = False,
) -> Dict[str, Any]:
    # Pass company_name to retrieve_chunks when use_hyde is True
    hits = retrieve_chunks(RAG_QUERY, top_k=top_k, filter_doc_id=doc["doc_id"], use_hyde=use_hyde, company_name=doc["company"])
    if not hits:
        return {"company": doc["company"], "error": f"No hits for {doc['doc_id']}"}

    user_prompt = build_extraction_prompt(doc["company"], RAG_QUERY, hits)
    t0 = time.time()
    raw = call_llm(system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:5000],
        }

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model
    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k": top_k,
        "use_hyde": use_hyde, # Add to metadata
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "file_name": h["file_name"],
            }
            for h in hits
        ],
    }
    return data

print("Updated `rag_extract_for_doc` function to support HyDE with company context in retrieval.")

Updated `rag_extract_for_doc` function to support HyDE with company context in retrieval.
